In [1]:
from PIL import Image,ImageDraw, ImageFont

In [6]:
path = "./raw_data/record_2022_1024_1100/vehicle.tesla.model3.master/"

In [7]:
img_path = path+"image_2/0000000218.png"

In [ ]:
img = Image.open(img_path)
img.show()

In [ ]:
img2 = Image.open(path+"lidar_image/0000000218.jpg")
img2 = img2.resize((1280,720))
img3 = Image.open(path+"image_2_semantic/0000000218.png")
img3 = img3.resize((860,540))
img4 = Image.open(path+"image_2_depth/0000000218.png")
img4 = img4.resize((860,540))
img5 = Image.open(path+"lidar_up_image/0000000218.jpg")
img5 = img5.resize((860,540))

In [ ]:
dst = Image.new('RGB', (img.width + img.width + 100, img.height+700))
dst.paste(img, (0, 0))
dst.paste(img2, (img.width+50, 0))
dst.paste(img3, (0,img.height+100))
dst.paste(img4,(img3.width+50,img.height+100))
dst.paste(img5,(img3.width+img4.width+100,img.height+100))

In [ ]:
dst.show()

In [ ]:
import matplotlib.pyplot as plt 
plt.axis('off')
plt.imshow(dst)

In [ ]:
import numpy as np
from vedo import Points,Plotter,show
import pandas as pd

In [8]:
numpy_points = np.load(path + "velodyne_semantic/0000000218.npy")

In [9]:
data = np.array(numpy_points.tolist())

In [11]:
numpy_points

array([(-13.873344 ,  1.7660547e+01,  6.017611 , -0., 98, 9, 0.9111924 ),
       (-13.805879 ,  1.7684204e+01,  6.011455 , -0., 98, 9, 0.9112791 ),
       (-13.7387495,  1.7708086e+01,  6.0054603, -0., 98, 9, 0.91136354),
       ...,
       ( -5.1969333, -4.7021482e-02, -2.4014132, -0., 98, 7, 0.9773597 ),
       ( -5.1970506, -3.1347655e-02, -2.4014132, -0., 98, 7, 0.9773597 ),
       ( -5.197119 , -1.5673827e-02, -2.4014132, -0., 98, 7, 0.9773597 )],
      dtype=[('x', '<f4'), ('y', '<f4'), ('z', '<f4'), ('v', '<f4'), ('ObjIdx', '<u4'), ('ObjTag', '<u4'), ('intensity', '<f4')])

In [10]:
data

array([[-1.38733444e+01,  1.76605473e+01,  6.01761103e+00, ...,
         9.80000000e+01,  9.00000000e+00,  9.11192417e-01],
       [-1.38058786e+01,  1.76842041e+01,  6.01145506e+00, ...,
         9.80000000e+01,  9.00000000e+00,  9.11279082e-01],
       [-1.37387495e+01,  1.77080860e+01,  6.00546026e+00, ...,
         9.80000000e+01,  9.00000000e+00,  9.11363542e-01],
       ...,
       [-5.19693327e+00, -4.70214821e-02, -2.40141320e+00, ...,
         9.80000000e+01,  7.00000000e+00,  9.77359712e-01],
       [-5.19705057e+00, -3.13476548e-02, -2.40141320e+00, ...,
         9.80000000e+01,  7.00000000e+00,  9.77359712e-01],
       [-5.19711924e+00, -1.56738274e-02, -2.40141320e+00, ...,
         9.80000000e+01,  7.00000000e+00,  9.77359712e-01]])

In [ ]:
RGBA = np.array([[0,0,0,255]]*data.shape[0])
pts = Points(data[:,0:3], r=2, c=RGBA)
plt = Plotter(interactive=False, offscreen=True)

In [ ]:
cam1 = dict(
    pos=(-18.43110, -455.6388, 208.1018),
    focalPoint=(-8.490190, 10.34274, 11.72185),
    viewup=(-0.07181808, 0.3886525, 0.9185812),
    distance=505.7694,
    clippingRange=(345.8786, 708.2590),
)
cam2 = dict(
    pos=(-18.43110, -455.6388, 208.1018),
    focalPoint=(-8.490190, 10.34274, 11.72185),
    viewup=(-0.07181808, 0.3886525, 0.9185812),
    distance=505.7694,
    clippingRange=(345.8786, 708.2590),
)

In [ ]:
plt.move_camera(camstart=cam1,camstop=cam2,fraction=0)
plt.screenshot("out1.png", scale=2).close()

In [ ]:
plt.show(pts)
cam3 = dict(
    pos=(-18.43110, -455.6388, 208.1018),
    focalPoint=(-8.490190, 10.34274, 11.72185),
    viewup=(-0.07181808, 0.3886525, 0.9185812),
    distance=505.7694,
    clippingRange=(345.8786, 708.2590),
)
show(pts, __doc__, axes=True,camera=cam3).close()

In [2]:
import open3d as o3d
from matplotlib import cm
import glob
import cv2
import os
from PIL import Image
import numpy as np
import pandas as pd

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
pcd = o3d.geometry.PointCloud()
vis = o3d.visualization.Visualizer()
vis.create_window(visible=False)

points = np.array([numpy_points['x'], numpy_points['y'], numpy_points['z']]).T
intensity = numpy_points['v']
intensity_col = intensity
cmap = cm.get_cmap('tab20c')
piyg = cmap(np.arange(0,cmap.N)) 
piyg_range = np.linspace(-15, 10, piyg.shape[0])
int_color = np.c_[
    np.interp(intensity_col, piyg_range, piyg[:, 0]),
    np.interp(intensity_col, piyg_range, piyg[:, 1]),
    np.interp(intensity_col, piyg_range, piyg[:, 2])]
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(int_color)
vis.add_geometry(pcd)
vis.get_render_option().background_color = [0,0,0]
quaternion = np.array([0, 0, 0, 3.14])
R = o3d.geometry.get_rotation_matrix_from_quaternion(quaternion)
    
view_control = vis.get_view_control()
front_vector = R[:, 0].tolist()
up_vector = R[:, 2].tolist()
# set up/lookat/front vector to vis
view_control.set_front(front_vector)
view_control.set_up(up_vector)
view_control.set_lookat(front_vector)
# because I want get first person view, so set zoom value with 0.001, if set 0, there will be nothing on screen.
view_control.set_zoom(0.01)
vis.capture_screen_image("file.jpg", do_render=True)
vis.destroy_window()

In [3]:
LABEL_COLORS = np.array([
    (255, 255, 255), # None
    (70, 70, 70),    # Building
    (100, 40, 40),   # Fences
    (55, 90, 80),    # Other
    (220, 20, 60),   # Pedestrian
    (153, 153, 153), # Pole
    (157, 234, 50),  # RoadLines
    (128, 64, 128),  # Road
    (244, 35, 232),  # Sidewalk
    (107, 142, 35),  # Vegetation
    (22, 196, 26),     # Vehicle
    (102, 102, 156), # Wall
    (220, 220, 0),   # TrafficSign
    (70, 130, 180),  # Sky
    (81, 0, 81),     # Ground
    (150, 100, 100), # Bridge
    (230, 150, 140), # RailTrack
    (180, 165, 180), # GuardRail
    (250, 170, 30),  # TrafficLight
    (110, 190, 160), # Static
    (170, 120, 50),  # Dynamic
    (45, 60, 150),   # Water
    (145, 170, 100), # Terrain
]) / 255.0 # normalize each channel [0-1] since is what Open3D uses

In [4]:
def text_3d(text, pos, direction=None, degree=0.0, font='arial.ttf', font_size=16):
    """
    Generate a 3D text point cloud used for visualization.
    :param text: content of the text
    :param pos: 3D xyz position of the text upper left corner
    :param direction: 3D normalized direction of where the text faces
    :param degree: in plane rotation of text
    :param font: Name of the font - change it according to your system
    :param font_size: size of the font
    :return: o3d.geoemtry.PointCloud object
    """
    if direction is None:
        direction = (0., 0., 1.)

    from PIL import Image, ImageFont, ImageDraw
    from pyquaternion import Quaternion

    font_obj = ImageFont.truetype(font, font_size)
    font_dim = font_obj.getsize(text)

    img = Image.new('RGB', font_dim, color=(255, 255, 255))
    
    draw = ImageDraw.Draw(img)
    draw.text((0, 0), text, font=font_obj, fill=(0, 255, 0))
    img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    img = np.asarray(img)
    img_mask = img[:, :, 0] < 128
    indices = np.indices([*img.shape[0:2], 1])[:, img_mask, 0].reshape(3, -1).T

    pcd = o3d.geometry.PointCloud()
    pcd.colors = o3d.utility.Vector3dVector(img[img_mask, :].astype(float) / 255.0)
    pcd.points = o3d.utility.Vector3dVector(indices / 100.0)

    raxis = np.cross([0.0, 0.0, 1.0], direction)
    if np.linalg.norm(raxis) < 1e-6:
        raxis = (0.0, 0.0, 1.0)
    trans = (Quaternion(axis=raxis, radians=np.arccos(direction[2])) *
             Quaternion(axis=direction, degrees=degree)).transformation_matrix
    trans[0:3, 3] = np.asarray(pos)
    pcd.transform(trans)
    return pcd

def get_target_point_and_bound(df,index):
    actor_points = df[df.ObjIdx == index].values
    actor_tag = actor_points[0][5]
    actor_speed = df[df.ObjIdx == index].v.mean()
    actor_points = np.asarray(actor_points[:,0:3])
    actor_pcd = o3d.geometry.PointCloud()
    actor_pcd.points =o3d.utility.Vector3dVector(actor_points)
    if len(actor_points) < 4:
        return None
    aabb = actor_pcd.get_axis_aligned_bounding_box()#.get_oriented_bounding_box()
    if actor_tag == 10:
        aabb.color = (1,0,0)
    elif actor_tag == 4:
        aabb.color = (0,1,0)
    else:
        return None
    return [aabb,text_3d("{:.2f}".format(actor_speed),aabb.get_center(),(2,0,0),degree=10,font_size=20)]

In [22]:
def get_aabb_bounds(numpy_cloud):
    df = pd.DataFrame(numpy_cloud)
    res = []
    target_ids = df[(df.ObjTag == 4) | (df.ObjTag == 10)].ObjIdx.unique()
    for i in target_ids:
        bound = get_target_point_and_bound(df,i)
        if bound is None:
            continue
        res.extend(bound)
    return res

def numpy_to_o3d(numpy_cloud,pcd,typ="velocity"):
    points = np.array([numpy_cloud['x'], numpy_cloud['y'], numpy_cloud['z']]).T
    if typ == "velocity" or typ == "up":
        intensity = numpy_cloud['v']
        intensity_col = intensity
        cmap = cm.get_cmap('tab20c')
        piyg = cmap(np.arange(0,cmap.N)) 
        piyg_range = np.linspace(-15, 10, piyg.shape[0])
        int_color = np.c_[
            np.interp(intensity_col, piyg_range, piyg[:, 0]),
            np.interp(intensity_col, piyg_range, piyg[:, 1]),
            np.interp(intensity_col, piyg_range, piyg[:, 2])]
        
    elif typ == "semantic":
        labels = np.array(numpy_cloud['ObjTag'])
        int_color = LABEL_COLORS[labels]
    elif typ == "length":
        df = pd.DataFrame(numpy_cloud)
        df["length"] = np.linalg.norm(df[["x", "y", "z"]], axis=1)
        length = df["length"]
        cmap = cm.get_cmap('hsv')
        piyg = cmap(np.arange(0,cmap.N)) 
        piyg_range = np.linspace(0, 80, piyg.shape[0])
        int_color = np.c_[
            np.interp(length, piyg_range, piyg[:, 0]),
            np.interp(length, piyg_range, piyg[:, 1]),
            np.interp(length, piyg_range, piyg[:, 2])]
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(int_color)
    return pcd

def gen_lidar_image(path,typ):
    vis = o3d.visualization.Visualizer()
    pcd = o3d.geometry.PointCloud()
    vis.create_window(visible=False)
    # vis.get_render_option().background_color = [0.05, 0.05, 0.05]
    vis.get_render_option().point_size = 3
    vis.get_render_option().show_coordinate_frame = False
    vis.get_render_option().background_color = [0,0,0]
    files = sorted(glob.glob("{}/vehicle.tesla.model3.master/velodyne_semantic/*.npy".format(path)))
    frame = 0
    bounds = []
    print(path)
    for file in files:
        raw_pcd = np.load(file)
        pcd = numpy_to_o3d(raw_pcd,pcd,typ)
        if typ!="up":
            for i in bounds:
                vis.remove_geometry(i,False)
        if frame == 0:
            vis.add_geometry(pcd)
            if typ != "up":
                quaternion = np.array([0, 0.1, 0, 3])
                R = o3d.geometry.get_rotation_matrix_from_quaternion(quaternion)
                view_control = vis.get_view_control()
                front_vector = R[:, 0].tolist()
                up_vector = R[:, 2].tolist()
                # set up/lookat/front vector to vis
                view_control.set_front(front_vector)
                view_control.set_up(up_vector)
                view_control.set_lookat(front_vector)
                # because I want get first person view, so set zoom value with 0.001, if set 0, there will be nothing on screen.
                view_control.set_zoom(0.01)
        if typ !="up":
            bounds = get_aabb_bounds(raw_pcd)
            for i in bounds:
                vis.add_geometry(i,False)
        vis.update_geometry(pcd)
        vis.poll_events()
        vis.update_renderer()
        frame += 1
        if typ == "velocity":
            folder = "lidar_image"
        elif typ == "up":
            folder = "lidar_up_image"
        elif typ == "semantic":
            folder = "lidar_semantic_image"
        elif typ == "length":
            folder = "lidar_length_image"
        vis.capture_screen_image("{}/vehicle.tesla.model3.master/{}/{}.jpg".format(path,folder,file.split(".")[-2].split("\\")[-1]))
    vis.destroy_window()

In [41]:
gen_lidar_image("./raw_data/record_2022_1024_1100/","velocity")

In [ ]:
sorted(glob.glob("{}/vehicle.tesla.model3.master/velodyne_semantic/*.npy".format("./raw_data/record_2022_1024_1100/")))[0].split(".")[-2].split("\\")[-1]

In [13]:
def get_all_file_list(path):
    file_list = []
    glob_list = sorted(glob.glob("{}/vehicle.tesla.model3.master/velodyne_semantic/*.npy".format(path)))
    for i in glob_list:
        file_list.append(i.split(".")[-2].split("\\")[-1])
    return file_list

In [14]:
def gen_image(path,i):
    dst = Image.new('RGB', (2660, 1780))
    img = Image.open(path+"image_2/{}.png".format(i))
    img2 = Image.open(path+"lidar_image/{}.jpg".format(i))
    img2 = img2.resize((1280,720))
    img3 = Image.open(path+"lidar_semantic_image/{}.jpg".format(i))
    img3 = img3.resize((860,540))
    img4 = Image.open(path+"lidar_length_image/{}.jpg".format(i))
    img4 = img4.resize((860,540))
    img5 = Image.open(path+"lidar_up_image/{}.jpg".format(i))
    img5 = img5.resize((860,540))
    dst.paste(img, (0, 0))
    dst.paste(img2, (img.width+50, 0))
    dst.paste(img3, (0,img.height+100))
    dst.paste(img4,(img3.width+50,img.height+100))
    dst.paste(img5,(img3.width+img4.width+100,img.height+100))
    
    
    font = ImageFont.truetype("arial.ttf", 100)
    img_draw = ImageDraw.Draw(dst)
    img_draw.text((30,30), "Image", font=font, fill=(255,0,0))
    img_draw.text((1330,30), "Velocity", font=font, fill=(255,0,0))
    font2 = ImageFont.truetype("arial.ttf", 50)
    img_draw.text((350,1380), "Semantic", font=font2, fill=(255,0,0))
    img_draw.text((1250,1380), "Depth", font=font2, fill=(255,0,0))
    img_draw.text((2150,1380), "Lidar", font=font2, fill=(255,0,0))

    
    dst = img_draw._image.resize((1330,990))
    return dst
        

In [25]:
def gen_video(path):
    folder = os.path.exists("{}/vehicle.tesla.model3.master/lidar_image".format(path))
    if not folder:
        os.makedirs("{}/vehicle.tesla.model3.master/lidar_image".format(path)) 
    folder = os.path.exists("{}/vehicle.tesla.model3.master/lidar_up_image".format(path))
    if not folder:
        os.makedirs("{}/vehicle.tesla.model3.master/lidar_up_image".format(path)) 
    folder = os.path.exists("{}/vehicle.tesla.model3.master/lidar_semantic_image".format(path))
    if not folder:
        os.makedirs("{}/vehicle.tesla.model3.master/lidar_semantic_image".format(path))
    older = os.path.exists("{}/vehicle.tesla.model3.master/lidar_length_image".format(path))
    if not folder:
        os.makedirs("{}/vehicle.tesla.model3.master/lidar_length_image".format(path))
    print("生成俯视图")
    gen_lidar_image(path,"up")
    print("生成速度点云")
    gen_lidar_image(path,"velocity")
    print("生成距离点云")
    gen_lidar_image(path,"length")
    print("生成语义点云")
    gen_lidar_image(path,"semantic")
    fps = 10          # 视频帧率
    size = (1330,990) # 需要转为视频的图片的尺寸
    video = cv2.VideoWriter("{}.avi".format(path), cv2.VideoWriter_fourcc('I', '4', '2', '0'), fps, size)
    file_list = get_all_file_list(path)
    frame = 0
    for file in file_list:
        img = gen_image(path+"/vehicle.tesla.model3.master/",file)
        data = cv2.cvtColor(np.asarray(img),cv2.COLOR_RGB2BGR)
        video.write(data)
        frame += 1
        if frame %10 == 0:
            print("frame {} is done".format(frame))
    video.release()

In [24]:
gen_video("./raw_data/record_2022_1102_1322")

生成俯视图
./raw_data/record_2022_1102_1322
生成速度点云
./raw_data/record_2022_1102_1322
生成距离点云
./raw_data/record_2022_1102_1322
生成语义点云
./raw_data/record_2022_1102_1322
frame 10 is done
frame 20 is done
frame 30 is done
frame 40 is done
frame 50 is done
frame 60 is done
frame 70 is done
frame 80 is done
frame 90 is done
frame 100 is done
frame 110 is done
frame 120 is done
frame 130 is done
frame 140 is done
frame 150 is done
frame 160 is done
frame 170 is done
frame 180 is done
frame 190 is done
frame 200 is done
frame 210 is done
frame 220 is done
frame 230 is done
frame 240 is done
frame 250 is done
frame 260 is done
frame 270 is done
frame 280 is done
frame 290 is done
frame 300 is done
frame 310 is done
frame 320 is done
frame 330 is done
frame 340 is done
frame 350 is done
frame 360 is done
frame 370 is done
frame 380 is done
frame 390 is done
frame 400 is done
frame 410 is done
frame 420 is done
frame 430 is done
frame 440 is done
frame 450 is done
frame 460 is done
frame 470 is done
fram

In [40]:
ttf = ImageFont.load_default()
ttf.info

[]